In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [60]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [61]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

26

In [62]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [63]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [23]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query = query)

In [26]:
context = ""
for doc in data:
    context += doc.page_content + "\n"


In [28]:
llm = ChatOllama(model= "phi4-mini")

In [31]:
#### Chain - Context_generate | prompt | llm | strparser

In [64]:
def get_context(query: str):
    data = vector_store.similarity_search(query = query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context": context,
        "question": query
    }

In [65]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answers based on the context for user questions.
    If you don not know the answer then you can say 'I do not know.'
    Context: {context}
    Question: {question}
""")

In [37]:
rag_chain = get_context | prompt | llm

In [70]:
res = rag_chain.invoke("What is value of RBC count and is it in range?")

In [71]:
print(res.content)

The RBC count value is 3.80 - 4.80 mill/mm³, and the reported RBC count is 34.47 mill/mm³. The reported value is not within the normal range provided.
